In [ ]:
!pip install -q -U datasets transformers trl peft accelerate bitsandbytes
!pip uninstall -y torchao

torchao is an optional PyTorch optimization/quantization package. Our GRPO practical does not need it. But PEFT detects the old installed torchao version and throws a compatibility error. So we remove torchao to avoid this conflict.

In [ ]:
import re
import torch
from difflib import SequenceMatcher
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, PeftModel
from trl import GRPOConfig, GRPOTrainer

| Code component         | Story character                                    |
| ---------------------- | -------------------------------------------------- |
| `SmolLM-135M-Instruct` | The AI student                                     |
| Dataset prompts        | Exam questions                                     |
| `reference_answer`     | Teacher’s answer key                               |
| Reward functions       | Three teachers/judges                              |

In [ ]:
data = [
    {
        "prompt": "Explain reinforcement learning in simple terms.",
        "reference_answer": "Reinforcement learning is a type of machine learning where an agent learns by taking actions, receiving rewards or penalties, and improving over time.",
    },
    {
        "prompt": "Write a polite email for delayed delivery.",
        "reference_answer": "We apologize for the delay in your delivery. Your order is on the way, and we appreciate your patience and understanding.",
    },
    {
        "prompt": "What is a neural network?",
        "reference_answer": "A neural network is a machine learning model inspired by the human brain. It learns patterns from data using layers of connected nodes.",
    },
    {
        "prompt": "Explain overfitting in ML.",
        "reference_answer": "Overfitting happens when a model learns the training data too closely, including noise, and performs poorly on new unseen data.",
    },
    {
        "prompt": "Explain supervised learning in simple terms.",
        "reference_answer": "Supervised learning is a machine learning method where a model learns from labeled examples and then predicts outputs for new data.",
    },
    {
        "prompt": "What is gradient descent?",
        "reference_answer": "Gradient descent is an optimization method that helps a model reduce its error by slowly adjusting its parameters in the right direction.",
    },
    {
        "prompt": "Explain classification in machine learning.",
        "reference_answer": "Classification is a machine learning task where the model assigns input data to predefined categories or classes.",
    },
    {
        "prompt": "Write a simple apology message.",
        "reference_answer": "I am sorry for the inconvenience. Thank you for your patience and understanding.",
    },
]

Prompt goes to model

↓

Model generates answers

↓

Reward function compares answers with reference_answer

↓

Good answer gets high reward

↓

Bad answer gets low reward

↓

GRPO updates model


In GRPO, our dataset mainly provides prompts. The model generates multiple answers for each prompt, then reward functions compare those answers with reference answers and give scores. GRPO does not directly copy the reference answer like SFT; it learns by getting higher rewards for better completions.

In [ ]:
def format_prompt(example):
    return {
        "prompt": f"Question: {example['prompt']}\nAnswer in simple English:"
    }

In [ ]:
dataset = Dataset.from_list(data)
dataset = dataset.map(format_prompt)

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

Dataset({
    features: ['prompt', 'reference_answer'],
    num_rows: 8
})

In [ ]:
dataset

{
    'prompt': 'Question: Explain reinforcement learning in simple terms.\nAnswer in simple English:',
    'reference_answer': 'Reinforcement learning is a type of machine learning where an agent learns by taking actions, receiving rewards or penalties, and improving over time.'
}

In [ ]:
model_id = "HuggingFaceTB/SmolLM-135M-Instruct"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_id)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "left"

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)

model.config.pad_token_id = tokenizer.pad_token_id

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

In [ ]:
lora_config = LoraConfig(
    task_type="CAUSAL_LM",
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
)

1. Rule-based/verifiable reward

2. Reward model

3. Existing/built-in reward functions

| Approach                                    | What it means                                                            | Where it is used                                      | Example                                                          | Practical meaning                                                     |
| ------------------------------------------- | ------------------------------------------------------------------------ | ----------------------------------------------------- | ---------------------------------------------------------------- | --------------------------------------------------------------------- |
| **1. Rule-based / Verifiable Reward**       | We write clear rules to check whether the model output is correct or not | Math, coding, JSON/XML format, exact-answer tasks     | Final answer is correct or not, code test cases passed or failed | Simple and reliable when the answer can be verified                   |
| **2. Reward Model**                         | A separate model gives a score to the generated output                   | Open-ended answers, helpfulness, preference alignment | How helpful, safe, clear, or preferred the answer is             | Human preference is converted into a model-based score                |
| **3. Built-in / Reusable Reward Functions** | We use existing reward functions or reusable custom functions            | Demos, standard tasks, formatting, accuracy checks    | Accuracy reward, format reward, length reward                    | We do not need to write every reward function from scratch every time |


| Component                | PPO                        | GRPO                                                        |
| ------------------------ | -------------------------- | ----------------------------------------------------------- |
| **Policy model**         | Present                    | Present                                                     |
| **Reward model**         | Present                    | can be used                                       |
| **Reference model**      | Present                    | Present                                                     |
| **Value / Critic model** | Present                    | **Not required**                                            |
| **Baseline**             | Comes from the value model | Comes from the average reward of multiple generated answers |


In [ ]:
#Text cleaning
def normalize_text(text: str) -> str:
    text = str(text).lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text

"   Reinforcement   Learning IS useful. "

"reinforcement learning is useful."

In [ ]:
#This function safely extracts the answer text from either a normal string or chat-message format, and converts unknown formats into a string.
def get_completion_text(completion):
    """
    Works for normal string completions.
    Also safe if completion comes in chat/message format.
    """
    if isinstance(completion, str):
        return completion

    if isinstance(completion, list):
        try:
            return completion[-1]["content"]
        except Exception:
            return str(completion)

    return str(completion)

In [ ]:
#It cleans both texts, compares their character/word sequence using SequenceMatcher, and returns a similarity score from 0 to 1—1 means identical and 0 means completely different.
def similarity_score(a: str, b: str) -> float:
    return SequenceMatcher(None, normalize_text(a), normalize_text(b)).ratio()

In [ ]:
# This function extracts unique words of at least 3 letters from both texts and returns the fraction of reference-text words (b) that also appear in the generated text (a).

# Example:

# a = "Machine learning uses data"
# b = "Machine learning learns from data"

# Common words: machine, learning, data

# So the score is:

# 3 common words ÷ 5 reference words = 0.6

def keyword_overlap_score(a: str, b: str) -> float:
    a_words = set(re.findall(r"[a-zA-Z]{3,}", normalize_text(a)))
    b_words = set(re.findall(r"[a-zA-Z]{3,}", normalize_text(b)))

    if not b_words:
        return 0.0

    return len(a_words & b_words) / len(b_words)

In [ ]:
# It acts like a teacher that compares the model’s answer with the answer key and gives marks out of 5.
def correctness_reward(prompts, completions, reference_answer=None, **kwargs):
    """
    Rewards answer similarity with reference answer.
    This is okay for demo, but not ideal for real production RL.
    """
    rewards = []

    if reference_answer is None:
        return [0.0 for _ in completions]

    for completion, ref in zip(completions, reference_answer):
        text = get_completion_text(completion)

        sim = similarity_score(text, ref)
        overlap = keyword_overlap_score(text, ref)

        # Combined correctness reward: 0 to 5
        score = (2.5 * sim) + (2.5 * overlap)
        rewards.append(float(score))

    return rewards

In [ ]:
# It acts like a teacher who rewards complete and properly written answers and penalizes short, vague, or unhelpful answers.
def helpfulness_reward(prompts, completions, **kwargs):
    """
    Rewards answers that look useful and complete.
    Penalizes vague/unhelpful responses.
    """
    rewards = []

    bad_phrases = [
        "i don't know",
        "no idea",
        "maybe",
        "not sure",
        "random magic",
    ]

    for completion in completions:
        text = get_completion_text(completion)
        text_norm = normalize_text(text)

        score = 0.0

        # Good answer length
        if 40 <= len(text) <= 260:
            score += 1.0
        elif len(text) < 20:
            score -= 1.0
        elif len(text) > 350:
            score -= 0.5

        # Sentence-like response
        if "." in text or "," in text:
            score += 0.5

        # Penalize vague responses
        if any(bp in text_norm for bp in bad_phrases):
            score -= 1.5

        rewards.append(float(score))

    return rewards

In [ ]:
# It acts like a teacher who rewards readable, properly sized answers and penalizes unclear or repetitive output.
def clarity_reward(prompts, completions, **kwargs):
    """
    Rewards readable English-style answers.
    """
    rewards = []

    for completion in completions:
        text = get_completion_text(completion)
        text_norm = normalize_text(text)

        score = 0.0

        # Has alphabetic content
        if re.search(r"[A-Za-z]", text):
            score += 0.5

        # Not too short / not too long
        if 30 <= len(text) <= 280:
            score += 0.5
        else:
            score -= 0.5

        # Penalize repeated junk
        if re.search(r"(.)\1{8,}", text_norm):
            score -= 1.0

        rewards.append(float(score))

    return rewards

In GRPO, the model does not directly learn from a fixed label like SFT. It generates multiple answers, and then reward functions score those answers. This code is only creating those reward functions: correctness checks similarity with the reference answer, helpfulness checks whether the answer is useful, and clarity checks whether the answer is readable. GRPOTrainer uses these scores to improve the model.

In GRPO, reward function is the teacher signal. For a demo, we can manually write simple rewards like correctness, helpfulness, and clarity. But in real systems, we do not keep writing unlimited reward functions manually. We usually use verifiable rewards, test cases, exact answer matching, LLM-as-judge, or a trained reward model. GRPO only needs a reward score; how we produce that reward depends on the task

In [ ]:
training_args = GRPOConfig(
    output_dir="grpo_output",

    learning_rate=1e-5,

    # Important:
    # In TRL GRPO, per_device_train_batch_size should be divisible by num_generations.
    # So we keep both as 4 for simple single-GPU Colab demo.
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    num_generations=4,

    max_completion_length=80,

    temperature=0.9,
    top_p=0.95,

    # KL penalty. 0.0 is default in recent TRL GRPO, but 0.01 is okay for teaching demo.
    beta=0.01,

    max_steps=10,

    logging_steps=1,
    save_steps=10,

    remove_unused_columns=False,
    report_to="none",

    gradient_checkpointing=False,

    fp16=torch.cuda.is_available(),
)

In [ ]:
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        correctness_reward,
        helpfulness_reward,
        clarity_reward,
    ],
    args=training_args,
    train_dataset=dataset,
    peft_config=lora_config,
)

In [ ]:
import trl
import inspect
from trl import GRPOConfig

print("TRL version:", trl.__version__)
print(inspect.signature(GRPOConfig.__init__))

TRL version: 1.7.0
(self, output_dir: str | None = None, per_device_train_batch_size: int = 8, num_train_epochs: float = 3.0, max_steps: int = -1, learning_rate: float = 1e-06, lr_scheduler_type: transformers.trainer_utils.SchedulerType | str = 'linear', lr_scheduler_kwargs: dict | str | None = None, warmup_steps: float = 0, optim: transformers.training_args.OptimizerNames | str = 'adamw_torch_fused', optim_args: str | None = None, weight_decay: float = 0.0, adam_beta1: float = 0.9, adam_beta2: float = 0.999, adam_epsilon: float = 1e-08, optim_target_modules: None | str | list[str] = None, gradient_accumulation_steps: int = 1, average_tokens_across_devices: bool = True, max_grad_norm: float = 1.0, label_smoothing_factor: float = 0.0, bf16: bool | None = None, fp16: bool = False, bf16_full_eval: bool = False, fp16_full_eval: bool = False, tf32: bool | None = None, gradient_checkpointing: bool = True, gradient_checkpointing_kwargs: dict[str, typing.Any] | str | None = None, torch_compile

In [ ]:
trainer.train()

Step,Training Loss
1,-0.000000
2,0.000000
3,0.000000
4,0.299315
5,0.160381
6,-0.008093
7,0.028336
8,-0.000001
9,0.000000
10,0.153412


TrainOutput(global_step=10, training_loss=0.06333511250093551, metrics={'train_runtime': 65.4125, 'train_samples_per_second': 0.612, 'train_steps_per_second': 0.153, 'total_flos': 0.0, 'train_loss': 0.06333511250093551, 'epoch': 1.25})

In [ ]:
trainer.save_model("grpo_output_final")
tokenizer.save_pretrained("grpo_output_final")

('grpo_output_final/tokenizer_config.json',
 'grpo_output_final/chat_template.jinja',
 'grpo_output_final/tokenizer.json')

In [ ]:
base_model_id = "HuggingFaceTB/SmolLM-135M-Instruct"
adapter_path = "grpo_output_final"

tokenizer = AutoTokenizer.from_pretrained(adapter_path)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)

model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(49152, 576, padding_idx=2)
        (layers): ModuleList(
          (0-29): 30 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=576, out_features=576, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=576, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=576, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Line

In [ ]:
def generate_answer(question, max_new_tokens=100):
    prompt = f"Question: {question}\nAnswer in simple English:"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
print(generate_answer("Explain AI in simple terms."))

Question: Explain AI in simple terms.
Answer in simple English:

**What is Artificial Intelligence (AI)?**

Artificial Intelligence, or AI, refers to the development of computer systems that can perform tasks that typically require human intelligence, such as:

1. **Learning**: AI systems can learn from data, improve their performance over time, and adapt to new situations.
2. **Reasoning**: AI systems can make decisions, solve problems, and draw conclusions based on data and rules.
3. **Perception**: AI systems can interpret


## GRPO with unsloth

In [ ]:
!pip install -U unsloth vllm
!pip install -U transformers datasets trl accelerate peft bitsandbytes

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024
lora_rank = 16

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    fast_inference=True,
    max_lora_rank=lora_rank,
    gpu_memory_utilization=0.6,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=lora_rank,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=lora_rank,
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

In [ ]:
SYSTEM_PROMPT = """
Respond in the following format:

<reasoning>
Write your step-by-step reasoning here.
</reasoning>
<answer>
Write only the final numeric answer here.
</answer>
"""

XML_COT_FORMAT = """\
<reasoning>
{reasoning}
</reasoning>
<answer>
{answer}
</answer>
"""

In [ ]:
import re
from datasets import load_dataset, Dataset

def extract_hash_answer(text: str):
    if "####" not in text:
        return None
    return text.split("####")[-1].strip()

def get_gsm8k_questions(split="train") -> Dataset:
    data = load_dataset("openai/gsm8k", "main")[split]

    data = data.map(
        lambda x: {
            "prompt": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": x["question"]},
            ],
            "answer": extract_hash_answer(x["answer"]),
        }
    )

    return data

dataset = get_gsm8k_questions("train")

print(dataset[0])

In [ ]:
def extract_xml_answer(text: str) -> str:
    if "<answer>" not in text:
        return ""
    answer = text.split("<answer>")[-1]
    answer = answer.split("</answer>")[0]
    return answer.strip()

In [ ]:
def correctness_reward_func(prompts, completions, answer, **kwargs) -> list[float]:
    responses = [completion[0]["content"] for completion in completions]
    extracted_responses = [extract_xml_answer(r) for r in responses]

    rewards = []
    for model_answer, gold_answer in zip(extracted_responses, answer):
        if model_answer == gold_answer:
            rewards.append(2.0)
        else:
            rewards.append(0.0)

    return rewards

In [ ]:
def int_reward_func(completions, **kwargs) -> list[float]:
    responses = [completion[0]["content"] for completion in completions]
    extracted_responses = [extract_xml_answer(r) for r in responses]

    return [0.5 if r.isdigit() else 0.0 for r in extracted_responses]

In [ ]:
def soft_format_reward_func(completions, **kwargs) -> list[float]:
    pattern = r"<reasoning>.*?</reasoning>\s*<answer>.*?</answer>"
    responses = [completion[0]["content"] for completion in completions]
    matches = [re.search(pattern, r, re.DOTALL) for r in responses]

    return [0.5 if match else 0.0 for match in matches]

In [ ]:
def strict_format_reward_func(completions, **kwargs) -> list[float]:
    pattern = r"^<reasoning>\n.*?\n</reasoning>\n<answer>\n.*?\n</answer>\n$"
    responses = [completion[0]["content"] for completion in completions]
    matches = [re.match(pattern, r, re.DOTALL) for r in responses]

    return [0.5 if match else 0.0 for match in matches]

In [ ]:
def count_xml(text) -> float:
    count = 0.0

    if text.count("<reasoning>\n") == 1:
        count += 0.125

    if text.count("\n</reasoning>\n") == 1:
        count += 0.125

    if text.count("\n<answer>\n") == 1:
        count += 0.125
        count -= len(text.split("\n</answer>\n")[-1]) * 0.001

    if text.count("\n</answer>") == 1:
        count += 0.125
        count -= (len(text.split("\n</answer>")[-1]) - 1) * 0.001

    return count

def xmlcount_reward_func(completions, **kwargs) -> list[float]:
    contents = [completion[0]["content"] for completion in completions]
    return [count_xml(c) for c in contents]

In [ ]:
from trl import GRPOConfig, GRPOTrainer

max_prompt_length = 256

training_args = GRPOConfig(
    learning_rate=5e-6,
    adam_beta1=0.9,
    adam_beta2=0.99,
    weight_decay=0.1,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",

    logging_steps=1,

    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,

    num_generations=4,  # increase to 6 if GPU memory allows

    max_prompt_length=max_prompt_length,
    max_completion_length=max_seq_length - max_prompt_length,

    max_steps=100,       # demo; use 250+ for better result
    save_steps=100,

    max_grad_norm=0.1,
    report_to="none",
    output_dir="grpo_outputs",
)

In [ ]:
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        xmlcount_reward_func,
        soft_format_reward_func,
        strict_format_reward_func,
        int_reward_func,
        correctness_reward_func,
    ],
    args=training_args,
    train_dataset=dataset,
)

In [ ]:
trainer.train()

In [ ]:
model.save_lora("grpo_saved_lora")

In [ ]:
from vllm import SamplingParams

test_prompt = tokenizer.apply_chat_template(
    [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": "If there are 3 boxes and each box has 4 apples, how many apples are there in total?"},
    ],
    tokenize=False,
    add_generation_prompt=True,
)

sampling_params = SamplingParams(
    temperature=0.7,
    top_p=0.95,
    max_tokens=512,
)

output = model.fast_generate(
    test_prompt,
    sampling_params=sampling_params,
    lora_request=model.load_lora("grpo_saved_lora"),
)[0].outputs[0].text

print(output)

In [ ]:
model.save_pretrained_merged(
    "grpo_merged_model",
    tokenizer,
    save_method="merged_16bit"
)